# От эксперимента к воспроизводимому результату

Практика 01, единое занятие. Первые 15 минут: запуск и краткий разбор готового эксперимента. Адаптация вводного taxi-примера MLOps Zoomcamp.

**Задача:** прогноз длительности поездки по двум зонам. Предполагаем, что пользователь сообщил зону назначения до старта. Фактические время окончания, длина поездки и стоимость не доступны на момент прогноза.

В комплекте реальные фиксированные подвыборки NYC TLC Green Taxi: январь 2021 — обучение, февраль — валидация. Это учебная историческая выборка, не оценка качества сервиса в 2026 году. Полный источник и хеши в `data/manifest.json`.

Ноутбук — готовая стартовая точка; далее заполните три TODO в `starter/src/taxi_duration` и добавьте два теста. CLI и сериализация уже готовы. Полного решения пакета в студенческом репозитории нет.

In [9]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

# Все пути ниже считаем от рабочей папки ядра; ожидаем папку practice.
ROOT = Path.cwd()
assert (ROOT / 'data/manifest.json').exists(), 'Откройте ноутбук из папки практики'
print('Python:', sys.version.split()[0])

Python: 3.13.12


## 1. Проверяем происхождение данных

Общий URL не фиксирует содержимое: сначала сверим хеши файлов с манифестом. Случайный отбор сделан заранее, до фильтрации длительности. Не скачиваем месяцы во время пары.

In [10]:
# Манифест хранит описание выборки и ожидаемые SHA-256 файлов.
manifest = json.loads((ROOT / 'data/manifest.json').read_text())
# Сверяем содержимое файлов с манифестом: при несовпадении остановимся.
# Это проверка версии данных, а не их качества или достоверности источника.
for name, info in manifest['files'].items():
    digest = hashlib.sha256((ROOT / 'data' / name).read_bytes()).hexdigest()
    assert digest == info['sha256'], f'Файл изменён: {name}'

# Загружаем подготовленные подвыборки: январь — train, февраль — validation.
train_raw = pd.read_parquet(ROOT / 'data/train.parquet')
validation_raw = pd.read_parquet(ROOT / 'data/validation.parquet')
print('До очистки:', len(train_raw), len(validation_raw))
# Первые пять строк помогают познакомиться с колонками и значениями.
train_raw.head()

До очистки: 10000 10000


,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,ride_id
0,2021-01-01 00:06:00,2021-01-01 00:39:00,39,225,green-2021-01-40496
1,2021-01-01 00:18:00,2021-01-01 00:38:00,174,69,green-2021-01-40473
2,2021-01-01 00:20:00,2021-01-01 00:37:00,168,212,green-2021-01-40491
3,2021-01-01 00:23:00,2021-01-01 00:28:00,17,17,green-2021-01-40479
4,2021-01-01 00:31:14,2021-01-01 00:55:07,244,244,green-2021-01-10


## 2. Определяем целевую переменную и область оценки

Длительность вычисляем в минутах. Учебный критерий: от 1 до 60 включительно. Он выбирает историческую когорту для обучения/оценки, но не является фильтром запросов: при прогнозе истинная длительность неизвестна.

Задание: объясните, почему RMSE после такой фильтрации нельзя объявлять RMSE для всех поездок.

In [11]:
def labeled_cohort(frame):
    # Работаем с копией, чтобы не менять исходную таблицу.
    result = frame.copy()
    # Разность времён переводим из секунд в минуты — это наш target.
    result['duration'] = (
        result['lpep_dropoff_datetime'] - result['lpep_pickup_datetime']
    ).dt.total_seconds() / 60
    # Обучаемся и оцениваемся только на поездках от 1 до 60 минут включительно.
    return result.loc[result.duration.between(1, 60)].copy()

train = labeled_cohort(train_raw)
validation = labeled_cohort(validation_raw)
# Проверяем временное разделение: все поездки validation позже train.
assert train.lpep_pickup_datetime.max() < validation.lpep_pickup_datetime.min()
print('После очистки:', len(train), len(validation))
print('Удалено:', len(train_raw) - len(train), len(validation_raw) - len(validation))

После очистки: 9676 9587
Удалено: 324 413


## 3. Признаки — категории, а не величины

Зоны 10 и 20 не означают удвоение какого-то свойства. Превращаем ID в строки и кодируем категории. В ноутбуке используются уже подготовленные данные; строгая валидация дробных, бесконечных и некорректных ID уже дана в заготовке пакета. Не переносите упрощённое приведение ID из notebook на произвольные входы.

Вопрос: почему нельзя вызвать `fit_transform` отдельно на валидации?

In [12]:
FEATURES = ['PULocationID', 'DOLocationID']

def feature_records(frame):
    # Пропуски — отдельная категория -1; строки задают ID как категории, не числа.
    # Одна поездка превращается в словарь вида {'PULocationID': '10', 'DOLocationID': '20'}.
    return frame[FEATURES].fillna(-1).astype('int64').astype(str).to_dict(orient='records')

# Разреженная матрица хранит ненулевые элементы, не заполняя память нулями.
vectorizer = DictVectorizer(sparse=True)
# Словарь категорий строим только по train; validation использует те же столбцы.
X_train = vectorizer.fit_transform(feature_records(train))
X_validation = vectorizer.transform(feature_records(validation))
# Целевую длительность держим отдельно от входных признаков.
y_train = train.duration.to_numpy()
y_validation = validation.duration.to_numpy()
print('Train:', X_train.shape, 'Validation:', X_validation.shape)
print('Ненулевых элементов:', X_train.nnz)

Train: (9676, 468) Validation: (9587, 468)
Ненулевых элементов: 19352


## 4. Сначала базовый прогноз, затем модель

Baseline всегда предсказывает среднюю длительность **из train**. Сравниваем с линейной регрессией на одной и той же валидации. Не подбираем гиперпараметры по этому небольшому примеру.

In [13]:
# Простой ориентир: каждой поездке предсказываем среднюю длительность из train.
baseline = np.full(len(validation), y_train.mean())

# Обучаем регрессию на train и прогнозируем для более позднего месяца.
model = LinearRegression()
model.fit(X_train, y_train)
validation_predictions = model.predict(X_validation)

# RMSE измеряется в минутах; меньше — лучше. Baseline и модель сравниваем на validation.
metrics = {
    'baseline_rmse': float(root_mean_squared_error(y_validation, baseline)),
    'train_rmse': float(root_mean_squared_error(y_train, model.predict(X_train))),
    'validation_rmse': float(root_mean_squared_error(y_validation, validation_predictions)),
}
metrics

{'baseline_rmse': 12.140851531759317,
 'train_rmse': 9.74152754514459,
 'validation_rmse': 10.662964115651745}

## 5. Число — ещё не заключение

Разница train/validation не доказывает drift сама по себе. У нас два месяца, фильтр 1–60 минут, небольшая выборка, нет отдельного test-периода и нет оценки погрешности метрики.

Дополнительно: выпишите ограничения и найдите долю строк validation с неизвестной категорией по каждому признаку. Предложите поведение для нового района. Не переобучайте vectorizer на новых запросах.

In [14]:
# Используем те же преобразования ID, что и при подготовке признаков.
normalized_train = pd.DataFrame(feature_records(train))
normalized_validation = pd.DataFrame(feature_records(validation))
for feature in FEATURES:
    # Категории, которые модель могла увидеть при обучении.
    known = set(normalized_train[feature])
    # ~ инвертирует маску isin; среднее по True/False даёт долю неизвестных ID.
    unknown_share = (~normalized_validation[feature].isin(known)).mean()
    print(feature, 'доля неизвестных:', round(float(unknown_share), 4))

PULocationID доля неизвестных: 0.0029
DOLocationID доля неизвестных: 0.0005


## 6. Предсказание без целевой переменной

Вход `inference.csv` содержит только идентификатор поездки и две зоны. Не фильтруем запросы по длительности и не меняем порядок строк.

In [15]:
requests = pd.read_csv(ROOT / 'data/inference.csv')
# На входе только ID поездки и две зоны — истинная длительность не нужна.
assert set(requests.columns) == {'ride_id', *FEATURES}
# Не переобучаем vectorizer: используем transform и сохраняем ID и порядок запросов.
response = pd.DataFrame({
    'ride_id': requests.ride_id,
    'predicted_duration': model.predict(vectorizer.transform(feature_records(requests))),
})
# На каждый запрос нужен один конечный числовой ответ, без NaN и бесконечностей.
assert len(response) == len(requests)
assert np.isfinite(response.predicted_duration).all()
response.head()

,ride_id,predicted_duration
0,green-2021-02-2,19.911722
1,green-2021-02-12,10.703007
2,green-2021-02-14,20.072486
3,green-2021-02-23,12.262883
4,green-2021-02-32,12.057704


## 7. Передаём модель вместе с преобразованием

Ниже сохраняем **собственные** объекты во временную папку и проверяем загрузку. Pickle может выполнять код: никогда не загружайте незнакомый `model.pkl`.

Готовый CLI заготовки после заполнения TODO пишет постоянные `model.pkl`, `metrics.json`, `run.json` в явно заданный новый каталог.

In [16]:
import pickle
import tempfile

# Временная папка автоматически удалится после выхода из блока with.
with tempfile.TemporaryDirectory(prefix='taxi-notebook-') as directory:
    path = Path(directory) / 'model.pkl'
    # Сохраняем и модель, и обученный кодировщик: важен тот же порядок признаков.
    with path.open('wb') as stream:
        pickle.dump((vectorizer, model), stream)
    # Загружаем только свой файл: pickle из недоверенного источника может выполнить код.
    with path.open('rb') as stream:
        restored_vectorizer, restored_model = pickle.load(stream)
    restored = restored_model.predict(restored_vectorizer.transform(feature_records(requests)))
    # После сохранения и загрузки прогнозы должны совпасть с числовым допуском.
    np.testing.assert_allclose(restored, response.predicted_duration)
print('Повторно загруженная модель даёт те же предсказания')

Повторно загруженная модель даёт те же предсказания


## 8. Передача эксперимента коллеге

1. Restart kernel → Run All: результат не зависит от скрытого порядка ячеек.
2. Зафиксируйте хеши входов, версии библиотек, правила выборки, метрики и ограничения.
3. Перейдите в starter: заполните TODO для duration, fit/transform и inference. Валидация уже реализована.
4. Добавьте два своих теста, выполните готовый CLI и вместе проверьте wheel вне исходников. Тайминг и команды — в README.